# Multi-Class Melanoma Discriminative Motif Results

This notebook reads saved outputs from `multi_class/runtime` only. Run `python multi_class/run_analysis.py --variant all --method all` first to generate the complete result set.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'multi_class':
    PROJECT_ROOT = PROJECT_ROOT.parent

RUNTIME_DIR = PROJECT_ROOT / 'multi_class' / 'runtime'
VARIANTS = ['all_classes_keep_tumor', 'all_classes_remove_tumor', 'without_pp_remove_tumor']
METHODS = ['one_vs_all_union', 'one_vs_one_union']
RUNTIME_DIR

In [ ]:
def read_json(path):
    return json.loads(path.read_text()) if path.exists() else None

def read_csv(path, **kwargs):
    return pd.read_csv(path, **kwargs) if path.exists() else pd.DataFrame()

summary_rows = []
for variant in VARIANTS:
    for method in METHODS:
        output_dir = RUNTIME_DIR / variant / method
        metrics = read_json(output_dir / 'metrics.json')
        if metrics is None:
            continue
        summary_rows.append({'variant': variant, 'method': method, **metrics})

summary = pd.DataFrame(summary_rows)
if summary.empty:
    print(f'No completed outputs found under {RUNTIME_DIR}')
else:
    display(summary.sort_values(['variant', 'method']))

## Cohort And Preprocessing

In [ ]:
patient_class_path = PROJECT_ROOT / 'melanoma_tutorials' / 'pet_data' / 'Melanoma' / 'patient_class.csv'
patient_classes = pd.read_csv(patient_class_path, header=None, names=['patient_id', 'patient_class'])
display(patient_classes['patient_class'].value_counts().rename_axis('class').reset_index(name='patients'))

preprocessing = []
for variant in VARIANTS:
    path = RUNTIME_DIR / variant / 'artifact' / 'preprocessing_summary.json'
    data = read_json(path)
    if data is not None:
        preprocessing.append(data)
display(pd.DataFrame(preprocessing) if preprocessing else 'No tumor-removal preprocessing summaries found yet.')

## Performance

In [ ]:
if not summary.empty:
    metric_cols = ['variant', 'method', 'n_patients', 'n_motifs', 'accuracy', 'balanced_accuracy', 'roc_auc_ovr_macro', 'f1_macro', 'f1_weighted']
    display(summary[[col for col in metric_cols if col in summary.columns]].sort_values('balanced_accuracy', ascending=False))
    plt.figure(figsize=(10, 4))
    sns.barplot(data=summary, x='variant', y='balanced_accuracy', hue='method')
    plt.xticks(rotation=30, ha='right')
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

## Comparison Tuning And Motif Counts

In [ ]:
tuning_tables = []
selected_tables = []
for variant in VARIANTS:
    for method in METHODS:
        output_dir = RUNTIME_DIR / variant / method
        tuning = read_csv(output_dir / 'comparison_tuning.csv')
        if not tuning.empty:
            tuning['variant'] = variant
            tuning['method'] = method
            tuning_tables.append(tuning)
        selected = read_csv(output_dir / 'selected_motifs.csv')
        if not selected.empty:
            selected['variant'] = variant
            selected['method'] = method
            selected_tables.append(selected)

tuning_df = pd.concat(tuning_tables, ignore_index=True) if tuning_tables else pd.DataFrame()
selected_df = pd.concat(selected_tables, ignore_index=True) if selected_tables else pd.DataFrame()
display(tuning_df.sort_values(['variant', 'method', 'comparison']) if not tuning_df.empty else 'No tuning outputs found.')
if not selected_df.empty:
    display(selected_df.groupby(['variant', 'method']).agg(selected_motifs=('ID', 'nunique')).reset_index())

## Confusion Matrices

In [ ]:
for variant in VARIANTS:
    for method in METHODS:
        cm = read_csv(RUNTIME_DIR / variant / method / 'confusion_matrix.csv', index_col=0)
        if cm.empty:
            continue
        plt.figure(figsize=(4.5, 3.8))
        sns.heatmap(cm, annot=True, fmt='g', cmap='Blues')
        plt.title(f'{variant}\n{method}')
        plt.tight_layout()
        plt.show()

## Feature Importance

In [ ]:
for variant in VARIANTS:
    for method in METHODS:
        importance = read_csv(RUNTIME_DIR / variant / method / 'feature_importance.csv')
        if importance.empty:
            continue
        top = importance.head(20).copy()
        display(top[['ID', 'mean_importance', 'comparison_provenance', 'comparison_label_provenance']])
        plt.figure(figsize=(8, 5))
        sns.barplot(data=top, x='mean_importance', y=top['ID'].astype(str), color='#4C78A8')
        plt.title(f'Top motif importances: {variant} / {method}')
        plt.ylabel('Motif ID')
        plt.tight_layout()
        plt.show()